# Whole Dataset - Address Standardization for Census Geocoder

Standardizes addresses in chunked `1_whole_properties_for_geocoding_*.csv` and outputs `2_whole_properties_for_geocoding_standardized_*.csv` (≤100MB each for GitHub).

**Run extraction notebook first.**

In [1]:
import os
import re
import glob
import pandas as pd

# --- Config: whole dataset, chunked I/O ---
DATA_LABEL = "whole"
DATA_DIR = "../../data/1_derived"
INPUT_PATTERN = os.path.join(DATA_DIR, f"1_{DATA_LABEL}_properties_for_geocoding_*.csv")

def normalize_address_range(addr):
    if not addr:
        return addr
    addr = re.sub(r'^(\d+)-\d+(\s)', r'\1\2', addr)
    return addr

def standardize_address(addr):
    if pd.isnull(addr):
        return ""
    addr = str(addr).strip().upper()
    addr = re.sub(r'\s+', ' ', addr)
    addr = re.sub(r'[^\w\s,]', '', addr)
    for k, v in {
        " STREET ": " ST ", " AVENUE ": " AVE ", " ROAD ": " RD ", " BOULEVARD ": " BLVD ",
        " DRIVE ": " DR ", " COURT ": " CT ", " PLACE ": " PL ", " LANE ": " LN ",
        " TERRACE ": " TER ", " PARKWAY ": " PKWY ", " SQUARE ": " SQ ", " SUITE ": " STE ",
        " APARTMENT ": " APT ", " UNIT ": " UNIT ",
    }.items():
        addr = addr.replace(k, v)
    return addr

# Process each chunk with checkpoint saving
input_files = sorted(glob.glob(INPUT_PATTERN))
print(f"Found {len(input_files)} input chunk(s)")
total_saved = 0
for i, inp_path in enumerate(input_files):
    chunk_num = inp_path.split("_")[-1].replace(".csv", "")
    out_name = f"2_{DATA_LABEL}_properties_for_geocoding_standardized_{chunk_num}"
    out_path = os.path.join(DATA_DIR, f"{out_name}.csv")
    df = pd.read_csv(inp_path)
    df["full_address_std"] = df["full_address"].apply(normalize_address_range).apply(standardize_address)
    df = df.drop(columns=["full_address"], errors="ignore")
    df.to_csv(out_path, index=False)
    sz_mb = os.path.getsize(out_path) / 1e6
    total_saved += len(df)
    print(f"  {i+1}/{len(input_files)}: {os.path.basename(inp_path)} → {out_name}.csv ({len(df):,} rows, {sz_mb:.1f} MB)")
print(f"\nTotal: {total_saved:,} properties standardized")
df.head()

Found 11 input chunk(s)
  1/11: 1_whole_properties_for_geocoding_001.csv → 2_whole_properties_for_geocoding_standardized_001.csv (100,000 rows, 26.6 MB)
  2/11: 1_whole_properties_for_geocoding_002.csv → 2_whole_properties_for_geocoding_standardized_002.csv (100,000 rows, 25.8 MB)
  3/11: 1_whole_properties_for_geocoding_003.csv → 2_whole_properties_for_geocoding_standardized_003.csv (100,000 rows, 25.8 MB)
  4/11: 1_whole_properties_for_geocoding_004.csv → 2_whole_properties_for_geocoding_standardized_004.csv (100,000 rows, 25.9 MB)
  5/11: 1_whole_properties_for_geocoding_005.csv → 2_whole_properties_for_geocoding_standardized_005.csv (100,000 rows, 26.4 MB)
  6/11: 1_whole_properties_for_geocoding_006.csv → 2_whole_properties_for_geocoding_standardized_006.csv (100,000 rows, 25.9 MB)
  7/11: 1_whole_properties_for_geocoding_007.csv → 2_whole_properties_for_geocoding_standardized_007.csv (100,000 rows, 26.7 MB)
  8/11: 1_whole_properties_for_geocoding_008.csv → 2_whole_properties_for

,LeaseCompId,PropertyId,PropertyName,Tenant.Name,TenantIndustry,TenantNAICS,Market,Submarket,Address.DeliveryAddress,Suite,...,Address.County,Address.PostalCode,Address.RegionName,Address.Subdivision,Address.Country,IsVerified.RawValue,SpaceUse,LeaseSource,LeaseOrigin,full_address_std
0,114108670.0,6112597.0,NaN,NaN,NaN,NaN,Portland,Lane County,539 E 11th Ave,NaN,...,Lane,97401.0,NaN,OR,United States,False,Office/Medical,Third Party - Landlord Rep,CoStar Research,"539 E 11TH AVE, EUGENE, OR 97401, UNITED STATES"
1,114158566.0,721667.0,Nimbus Corporate Center - Building 6,Cascade Microtech Inc,Manufacturing,Semiconductor and Related Device Manufacturing,Portland,217 Corridor/Beaverton,9100 SW Gemini Dr,9203B 9205 9215 9225,...,Washington,97008.0,NaN,OR,United States,False,Office,NaN,CoStar Research,"9100 SW GEMINI DR, BEAVERTON, OR 97008, UNITED..."
2,115503251.0,9042216.0,Retail Building,Aaron's,Retailer,Electronics and Appliance Retailers,Portland,Lane County,3 River Ave,NaN,...,Lane,97404.0,NaN,OR,United States,False,Retail,Third Party - Other,CoStar Research,"3 RIVER AVE, EUGENE, OR 97404, UNITED STATES"
3,114524978.0,7087085.0,NaN,NaN,NaN,NaN,Portland,Lane County,745-749 Willamette St,NaN,...,Lane,97401.0,NaN,OR,United States,False,Retail,Third Party - Landlord Rep,CoStar Research,"745 WILLAMETTE ST, EUGENE, OR 97401, UNITED ST..."
4,114100694.0,1357615.0,Amberwood West,McKenzie Books Inc,Retailer,Book Retailers and News Dealers,Portland,Sunset Corridor/Hillsboro,21220 NW Amberwood Dr,NaN,...,Washington,97124.0,NaN,OR,United States,False,Industrial,Third Party - Landlord Rep,CoStar Research,"21220 NW AMBERWOOD DR, HILLSBORO, OR 97124, UN..."
